# SLR to exclude gender bias

- kernel: R 4.1.3, r_env
- desc:
  - Use simple linear regressive among all samples to exclude potential bias of gender
  - Note: This is the second SLR analysis. The first one was performed to remove aging-related species
    (considering only Case samples for abundance-increasing species and only Control samples for abundance-decreasing species).


## load modules

In [101]:
library(tidyverse)
library(ggplot2)
library(patchwork)
library(logging)

options(warn = -1)  # no warning message

source('../r_utils.r')

tb_dir <- '../tables/'
fig_dir <- '../figures/'

In [2]:
# taxonomic profile with clinical info
taxon_profile <- read_csv('../tables/taxonomic_clin.csv') %>% 
    filter(clade_type == 'species')
loginfo(
    "%d records, %d species of %d samples identified", 
    nrow(taxon_profile),
    n_unique(taxon_profile, 'clade_short'),
    n_unique(taxon_profile, 'sample')
)

# species from DA results
da_species <- read_csv('../tables/abundance-da-species.csv') %>% 
    filter(status != 'unchange') %>% 
    select('clade_name') %>% 
    merge(unique(select(taxon_profile, 'clade_name', 'clade_short'))) %>%
    pull('clade_short')
loginfo('%d species passed from DA results', length(da_species))

# species passed 1st slr: age irrelevant species
age_ir_species <- read_csv('../tables/abundance-slr-age.csv') %>% 
    filter(
        (slr.status == 'irrelevant') | 
       ((da.status == 'down') & (slr.status == 'positive')) |
       ((da.status == 'up') & (slr.status == 'negative'))
    ) %>% 
    pull(clade_short)
loginfo('%d species passed 1st SLR filtering (age-irrelevant)', length(age_ir_species))

2025-12-02 13:37:04 INFO::49938 records, 574 species of 87 samples identified
2025-12-02 13:37:04 INFO::21 species passed from DA results
2025-12-02 13:37:04 INFO::20 species passed 1st SLR filtering (age-irrelevant)


In [3]:
# n_species * n_samples should equal to nrow
574 * 87 == 49938

[1] TRUE

## slr: based on DA results

In [4]:
df_info <- taxon_profile %>% 
    select(sample, clade_short, relab, gender) %>%
    mutate(gender = factor(gender, levels = c('F', 'M')),)

In [5]:
# for each selected species
df_res <- dplyr::bind_rows()
for (clade in da_species) {
    subdf <- filter(df_info, clade_short == clade)
    res <- summary(lm(relab ~ gender, data = subdf))
    df_res <- dplyr::bind_rows(
        df_res, 
        data.frame(
            clade_short = clade,
            r.squared = res$r.squared,
            r.squared.adj = res$adj.r.squared,
            intercept = res$coefficients['(Intercept)', 'Estimate'],
            slope = res$coefficients['genderM', 'Estimate'],
            p = 1 - pf(res$fstatistic[1], res$fstatistic[2], res$fstatistic[3]),
            p.intercept = res$coefficients['(Intercept)', 'Pr(>|t|)'],
            p.slope = res$coefficients['genderM', 'Pr(>|t|)']
        )
    )
}

In [6]:
df_res <- df_res %>% 
    mutate(`slr.status` = case_when(
        p < 0.05 & slope < 0 ~ 'negative',
        p < 0.05 & slope > 0 ~ 'positive',
        TRUE ~ 'irrelevant'
    ))
df_res %>% write_csv(file.path(tb_dir, 'abundance-slr-genderM.csv'))

In [7]:
df_res %>% filter(slr.status != 'irrelevant')

clade_short,r.squared,r.squared.adj,intercept,slope,p,p.intercept,p.slope,slr.status
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
Klebsiella_pneumoniae,0.05355306,0.04228584,0.2288128,0.5621381,0.03204516,0.2129673,0.03204516,positive


In [8]:
df_res %>% filter(!clade_short %in% age_ir_species)

clade_short,r.squared,r.squared.adj,intercept,slope,p,p.intercept,p.slope,slr.status
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
Escherichia_coli,3.303402e-05,-0.01187133,2.766327,0.07689419,0.9581137,0.00885567,0.9581137,irrelevant
